# Capítulo 2 · Instrucciones que se pueden comprobar

El mismo agente de programación en el repositorio de Ciclos Norte, con cuatro versiones del mismo `AGENTS.md`: en prosa, como ocho reglas numeradas, con las reglas comprobadas al terminar y con las reglas de proceso comprobadas en el momento de editar. Al final de cada tarea, un programa comprueba las ocho reglas con el diff, el registro de llamadas y la respuesta.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 2: instrucciones que se pueden comprobar."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. El modelo

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con decodificación voraz. Devuelve el texto y los
    tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=MAX_TOKENS_PASO,
                                 do_sample=False, temperature=None, top_p=None, top_k=None,
                                 prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 3. Dos versiones del mismo AGENTS.md

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_PROSA = """# AGENTS.md · Ciclos Norte

Ciclos Norte es una tienda de bicicletas de Oviedo con tienda física, taller y venta
por internet. Este repositorio es su programa de gestión, que empezó como unos pocos
scripts para las devoluciones y ha ido creciendo con los años: hoy lleva los pedidos de
la web, los envíos, las devoluciones, el almacén, los presupuestos del taller, las
facturas y los avisos que reciben los clientes por correo. Lo mantiene un equipo
pequeño, así que agradecemos los cambios pequeños, bien explicados y fáciles de revisar.

El código está escrito en Python, solo con la biblioteca estándar, y cada módulo se
ocupa de una parte de la tienda. Las reglas del negocio, como el IVA, las tarifas de
envío o los plazos, están reunidas en config.py y salen de las condiciones generales de
venta, que están en docs/. Preferimos nombres en español, funciones cortas con su
docstring y que las reglas no se escriban a mano en más de un sitio. Antes de cambiar
un archivo conviene leerlo con calma, porque muchas funciones dependen de otras y lo
que parece un error a veces es una regla de la tienda.

Los datos de ejemplo de datos/ son la copia de la base de datos con la que probamos,
así que no hay que modificarlos nunca: si un dato parece mal, el problema suele estar
en el código que lo lee. La documentación de docs/ la revisa dirección, y solo se
cambia cuando la petición pide expresamente cambiar las condiciones generales.

Las pruebas son importantes para nosotros. Todas las pruebas de un módulo están en su
archivo de tests/ (tests/test_envios.py para envios.py, y así con todos), y cuando se
cambia algo hay que añadir una prueba que lo compruebe en ese mismo archivo, no en uno
nuevo. Para añadirla, se sustituye la línea final `if __name__ == "__main__":` por una
clase nueva con la prueba, seguida de esa misma línea, e importando dentro lo que haga
falta. Las pruebas que ya existen no se borran ni se cambian, salvo cuando la petición
cambia una regla del negocio: entonces hay que actualizar las que comprueban la regla
antigua y contarlo en la respuesta. Tampoco creamos módulos nuevos sin hablarlo antes.

Damos un trabajo por terminado cuando se ha cambiado el código del programa que decide
lo que pide la petición, hay una prueba nueva que lo comprueba y, después del último
cambio, se han ejecutado todas las pruebas y pasan. Al terminar, la respuesta debe
explicar lo que se ha hecho y nombrar todos los archivos que se han cambiado, para que
quien revise sepa dónde mirar. Si algo no se ha podido hacer, es mejor decirlo que dejar
un cambio a medias sin avisar.
"""


AGENTS_REGLAS = """# AGENTS.md · Ciclos Norte

Reglas del repositorio. Cada una se comprueba al final de tu trabajo.

R1. Antes de editar un archivo, léelo con leer_archivo.
R2. No cambies nada de datos/.
R3. Cambia docs/ solo si la petición pide cambiar las condiciones generales.
R4. No borres ni cambies pruebas que ya existen. Excepción: si la petición cambia una
    regla del negocio, actualiza las pruebas de la regla antigua y dilo en tu respuesta.
R5. Añade tus pruebas en el archivo del módulo (tests/test_envios.py para envios.py):
    sustituye la línea `if __name__ == "__main__":` por una clase nueva con tu prueba,
    seguida de esa misma línea. Importa dentro de la prueba lo que necesites.
R6. No crees módulos nuevos fuera de tests/.
R7. «TERMINADO» exige: cambios en el código del programa, una comprobación nueva
    en tests/ y ejecutar_pruebas en verde después de tu último cambio.
R8. Tu respuesta final nombra todos los archivos que has cambiado.
"""


PIDE_DOCS = re.compile(r"(actualiz\w*|y en las)\s[^.]{0,40}condiciones\s+generales", re.I)

In [ ]:
print(len(AGENTS_PROSA.split()), "palabras en prosa;", len(AGENTS_REGLAS.split()), "en reglas")

## 4. Las reglas, comprobadas sobre el diff

Se pueden probar sin modelo: aplicamos la solución de una tarea y comprobamos las reglas con una respuesta que no nombra los archivos.

In [ ]:
def lineas_del_diff(diff):
    """Las líneas añadidas y quitadas de cada archivo de un diff unificado:
    {ruta: {"+": [...], "-": [...]}}."""
    archivos, actual = {}, None
    for linea in diff.splitlines():
        if linea.startswith("+++ b/"):
            actual = archivos.setdefault(linea[6:], {"+": [], "-": []})
        elif linea.startswith("--- a/") or actual is None:
            continue
        elif linea[:1] in "+-" and linea[1:].strip():
            actual[linea[0]].append(linea[1:])
    return archivos


def comprobar_reglas(tarea, registro, cambios, respuesta):
    """Las reglas incumplidas, como {regla: evidencia}. Mira lo que de verdad ha
    cambiado (el diff), el registro de llamadas y la respuesta final."""
    rotas = {}
    lineas = lineas_del_diff(cambios["diff"])
    leidos = set()
    for r in registro:
        ruta = r["argumentos"].get("ruta", "")
        if r["herramienta"] == "leer_archivo" and r["ok"]:
            leidos.add(ruta)
        elif r["herramienta"] == "editar" and r["ok"] and ruta not in leidos:
            rotas.setdefault("R1", f"editaste {ruta} sin leerlo antes")
    for ruta in cambios["cambiados"]:
        if ruta.startswith("datos/"):
            rotas["R2"] = f"cambiaste {ruta}"
        if ruta.startswith("docs/") and not PIDE_DOCS.search(tarea["peticion"]):
            rotas["R3"] = f"cambiaste {ruta} y la petición no lo pide"
        if ruta.startswith("tests/") and ruta in cambios["nuevos"]:
            rotas["R5"] = f"creaste {ruta}; las pruebas van en el archivo del módulo"
        if not ruta.startswith(("tests/", "docs/", "datos/")) and ruta in cambios["nuevos"]:
            rotas["R6"] = f"creaste el módulo {ruta}"
    quitadas = [r for r, l in lineas.items() if r.startswith("tests/") and l["-"]]
    if quitadas and not re.search(r"prueba", respuesta or "", re.I):
        rotas["R4"] = (f"cambiaste pruebas que ya existían en {', '.join(quitadas)} "
                       "y tu respuesta no lo dice")
    if respuesta and dice_terminado(respuesta):
        programa = [r for r in cambios["cambiados"]
                    if not r.startswith(("tests/", "docs/", "datos/"))]
        prueba_nueva = any(re.search(r"def test_|assert", l)
                           for r, l2 in lineas.items() if r.startswith("tests/")
                           for l in l2["+"])
        ediciones = [i for i, r in enumerate(registro)
                     if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
        despues = registro[ediciones[-1] + 1:] if ediciones else registro
        pruebas = [r for r in despues if r["herramienta"] == "ejecutar_pruebas"]
        falta = [texto for cumple, texto in (
            (programa, "no hay cambios en el código del programa"),
            (prueba_nueva, "no hay ninguna comprobación nueva en tests/"),
            (pruebas and pruebas[-1]["resultado"].get("todas_pasan"),
             "no hay una ejecución de las pruebas en verde después del último cambio"))
            if not cumple]
        if falta:
            rotas["R7"] = "; ".join(falta)
        sin_nombrar = [r for r in cambios["cambiados"]
                       if Path(r).name not in respuesta]
        if sin_nombrar:
            rotas["R8"] = f"tu respuesta no nombra {', '.join(sin_nombrar)}"
    return rotas


def que_falta(registro):
    """La puerta del capítulo 1, con el registro de llamadas: un cambio en el código,
    una prueba nueva y las pruebas pasadas después."""
    ediciones = [i for i, r in enumerate(registro)
                 if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
    ruta = lambda i: registro[i]["argumentos"].get("ruta", "")  # noqa: E731
    falta = []
    if not any(not ruta(i).startswith("tests/") for i in ediciones):
        falta.append("no has cambiado el código del programa")
    if not any(ruta(i).startswith("tests/") for i in ediciones):
        falta.append("no has añadido ninguna prueba en tests/")
    ultima = ediciones[-1] if ediciones else -1
    pruebas = [r for r in registro[ultima + 1:] if r["herramienta"] == "ejecutar_pruebas"]
    if not pruebas:
        falta.append("no has ejecutado ejecutar_pruebas después de tu último cambio")
    elif not pruebas[-1]["resultado"].get("todas_pasan"):
        falta.append("la última ejecución de las pruebas tiene fallos")
    return falta


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)

In [ ]:
tarea = hr.leer_tarea("T15")
with hr.Espacio() as espacio:
    h = hr.Herramientas(espacio)
    hr.aplicar_solucion(h, tarea["carpeta"] / "solucion")
    h.llamar("ejecutar_pruebas")
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    print(cambios["cambiados"])
    print(comprobar_reglas(tarea, h.registro, cambios, "TERMINADO: hecho."))

## 5. Las reglas de proceso, antes de editar

In [ ]:
class HerramientasConReglas:
    """Las herramientas del agente con las reglas de proceso comprobadas antes de
    cada edición: si la edición incumple R1, R2, R3, R5 o R6, no se hace y el agente
    recibe la regla. El resto de llamadas pasan tal cual."""

    def __init__(self, herramientas, tarea):
        self.herramientas = herramientas
        self.registro = herramientas.registro
        self.tarea = tarea
        self.leidos = set()

    def regla_rota(self, nombre, ruta):
        """La regla que incumpliría editar o crear «ruta», o None."""
        existe = self.herramientas.espacio.ruta(ruta).exists()
        if ruta.startswith("datos/"):
            return "R2: no cambies nada de datos/."
        if ruta.startswith("docs/") and not PIDE_DOCS.search(self.tarea["peticion"]):
            return "R3: la petición no pide cambiar las condiciones generales."
        if nombre == "crear_archivo" and ruta.startswith("tests/"):
            return ("R5: no crees archivos de pruebas; añade tu prueba al archivo del "
                    "módulo, sustituyendo la línea if __name__ == \"__main__\":")
        if nombre == "crear_archivo" and not existe:
            return "R6: no crees módulos nuevos fuera de tests/."
        if nombre == "editar" and existe and ruta not in self.leidos:
            return f"R1: lee {ruta} con leer_archivo antes de editarlo."
        return None

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, pero comprobando las reglas antes de editar."""
        ruta = str((argumentos or {}).get("ruta", ""))
        if nombre in ("editar", "crear_archivo"):
            regla = self.regla_rota(nombre, ruta)
            if regla:
                resultado = {"error": "Edición rechazada por AGENTS.md. " + regla}
                self.registro.append({"herramienta": nombre, "argumentos": argumentos,
                                      "ok": False, "segundos": 0,
                                      "resultado": resultado})
                return resultado
        resultado = self.herramientas.llamar(nombre, argumentos)
        if nombre == "leer_archivo" and "error" not in resultado:
            self.leidos.add(ruta)
        return resultado

In [ ]:
with hr.Espacio() as espacio:
    h = HerramientasConReglas(hr.Herramientas(espacio), hr.leer_tarea("T02"))
    print(h.llamar("editar", {"ruta": "envios.py", "buscar": "a", "poner": "b"}))
    print(h.llamar("crear_archivo", {"ruta": "tests/test_cp.py", "contenido": "import unittest"}))

## 6. El agente

La tarea del IVA con las reglas comprobadas al final y al actuar. Busca el mensaje de la puerta con la R1 y la edición rechazada.

In [ ]:
def agente(tok, modelo, tarea, instrucciones, puerta="registro", al_editar=False,
           exigir_marca=False, max_pasos=MAX_PASOS, ver=False):
    """El bucle del agente con unas instrucciones y una puerta: «registro», la del
    capítulo 1; «reglas», que comprueba las reglas sobre el diff y devuelve las
    incumplidas, o «resultado», que solo devuelve las de resultado (R4, R7 y R8). Con «al_editar», las reglas de proceso se comprueban antes de cada
    edición. Con «exigir_marca», una respuesta sin herramientas que no empieza por
    TERMINADO ni NO TERMINADO no acaba la tarea: el agente sigue. Devuelve la fila de
    resultados, con las reglas incumplidas al final."""
    inicio = time.time()
    mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + instrucciones},
                {"role": "user", "content": tarea["peticion"]}]
    final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
    with hr.Espacio() as espacio:
        herramientas = hr.Herramientas(espacio)
        if al_editar:
            herramientas = HerramientasConReglas(herramientas, tarea)
        for pasos in range(1, max_pasos + 1):
            texto, recortados = recortar_conversacion(tok, mensajes)
            if recortados is None:
                break
            respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
            entrada, salida = entrada + n_entrada, salida + n_salida
            llamadas = leer_llamadas(respuesta)
            if not llamadas:
                mensajes.append({"role": "assistant", "content": respuesta})
                if ver:
                    print(f"[{pasos}] {respuesta}")
                if exigir_marca and "TERMINADO" not in respuesta:
                    mensajes.append({"role": "user", "content":
                                     "Sigue con las herramientas, o responde empezando "
                                     "por «TERMINADO:» o «NO TERMINADO:»."})
                    continue
                if dice_terminado(respuesta) and rechazos < MAX_RECHAZOS:
                    if puerta in ("reglas", "resultado"):
                        cambios = hr.diferencias(espacio.origen, espacio.carpeta)
                        rotas = comprobar_reglas(tarea, herramientas.registro, cambios,
                                                 respuesta)
                        if puerta == "resultado":   # solo las reglas de resultado
                            rotas = {r: texto for r, texto in rotas.items()
                                     if r in ("R4", "R7", "R8")}
                        falta = [f"{r}: {texto}" for r, texto in sorted(rotas.items())]
                    else:
                        falta = que_falta(herramientas.registro)
                    if falta:
                        rechazos += 1
                        aviso = ("Todavía no está terminado según AGENTS.md: "
                                 + "; ".join(falta) + ". Sigue trabajando.")
                        mensajes.append({"role": "user", "content": aviso})
                        if ver:
                            print(f"    PUERTA: {aviso}")
                        continue
                final = respuesta
                break
            pensamiento = respuesta.split("<tool_call>")[0].strip()
            mensajes.append({"role": "assistant", "content": pensamiento,
                             "tool_calls": [{"type": "function", "function": l}
                                            for l in llamadas if l["name"]]})
            for llamada in llamadas:
                if llamada["name"] is None:
                    resultado = {"error": "No se entiende la llamada a la herramienta."}
                else:
                    resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
                if ver:
                    print(f"[{pasos}] {llamada['name']}"
                          f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
                mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                                 "content": json.dumps(resultado, ensure_ascii=False)})
        evaluacion = hr.evaluar_intento(espacio, tarea)
        cambios = hr.diferencias(espacio.origen, espacio.carpeta)
        rotas = comprobar_reglas(tarea, herramientas.registro, cambios, final)
    terminado = final is not None and dice_terminado(final)
    return {"tarea": tarea["id"], "tipo": tarea["meta"]["tipo"],
            "resuelta": evaluacion["resuelta"], "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "reglas_rotas": rotas, "rechazos": rechazos,
            "ocultas": evaluacion["ocultas"], "visibles": evaluacion["visibles"],
            "fuera_de_alcance": evaluacion["fuera_de_alcance"],
            "pasos": pasos, "tokens_entrada": entrada, "tokens_salida": salida,
            "segundos": round(time.time() - inicio, 1),
            "respuesta": final or "(sin respuesta final)",
            "llamadas": [[r["herramienta"], r["argumentos"].get("ruta", ""), r["ok"]]
                         for r in herramientas.registro],
            "diff": evaluacion["diff"]}


VARIANTES = {
    "en prosa": {"instrucciones": AGENTS_PROSA},
    "reglas": {"instrucciones": AGENTS_REGLAS},
    "reglas comprobadas": {"instrucciones": AGENTS_REGLAS, "puerta": "reglas"},
    "reglas en el momento": {"instrucciones": AGENTS_REGLAS, "puerta": "reglas",
                             "al_editar": True},
}

In [ ]:
for nombre in ("reglas comprobadas", "reglas en el momento"):
    print(f"--- {nombre}")
    fila = agente(tok, modelo, hr.leer_tarea("T04"), ver=True, **VARIANTES[nombre])
    print("Resuelta:", fila["resuelta"], "· Reglas rotas:", fila["reglas_rotas"])

## 7. Las catorce tareas con las cuatro variantes

Por defecto se prueba una muestra; con `COMPLETO = True`, el experimento entero del libro.

In [ ]:
def resumen(filas):
    """Los totales de una variante, con las reglas incumplidas de cada tipo."""
    por_regla = {}
    for f in filas:
        for regla in f["reglas_rotas"]:
            por_regla[regla] = por_regla.get(regla, 0) + 1
    return {"tareas": len(filas),
            "resueltas": sum(f["resuelta"] for f in filas),
            "dice_terminado": sum(f["dice_terminado"] for f in filas),
            "terminado_falso": sum(f["terminado_falso"] for f in filas),
            "reglas_rotas": sum(len(f["reglas_rotas"]) for f in filas),
            "tareas_sin_reglas_rotas": sum(not f["reglas_rotas"] for f in filas),
            "por_regla": dict(sorted(por_regla.items())),
            "pasos": sum(f["pasos"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def probar(tok, modelo, opciones, tareas=TAREAS):
    """Una variante con todas las tareas: sus filas y su resumen."""
    filas = []
    for identificador in tareas:
        fila = agente(tok, modelo, hr.leer_tarea(identificador), **opciones)
        filas.append(fila)
        print(f"  {identificador} {'resuelta' if fila['resuelta'] else 'no      '}"
              f"  {'dice terminado' if fila['dice_terminado'] else 'no lo dice    '}"
              f"  reglas rotas {','.join(sorted(fila['reglas_rotas'])) or '-':12s}"
              f"  pasos={fila['pasos']:2d}  {fila['segundos']:6.1f} s", flush=True)
    return {"filas": filas, "resumen": resumen(filas)}

In [ ]:
# Con COMPLETO = False se prueban dos tareas (unos 35 minutos en la T4 de Colab);
# con True, el experimento entero del capítulo (unas catorce horas en la T4).
COMPLETO = False
tareas = TAREAS if COMPLETO else ["T04", "T17"]
resultados = {}
for nombre, opciones in VARIANTES.items():
    print(f"== {nombre}")
    resultados[nombre] = probar(tok, modelo, opciones, tareas=tareas)
    print(resultados[nombre]["resumen"])

Los resultados completos del libro, con todas las tareas, están guardados en `datos/resultados/capitulo_2/`, que `preparar` ha descargado:

In [ ]:
import json
guardados = json.loads((RAIZ / "datos" / "resultados" / "capitulo_2" / "resultados.json").read_text(encoding="utf-8"))
for nombre, r in guardados.items():
    print(nombre, r["resumen"])

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_2/`, que `preparar` descarga con los datos.

1. Para cada una de las reglas R1 a R8, di si es de proceso o de resultado, de qué fuente sale la evidencia (diff, registro o respuesta) y en qué momento se puede comprobar. *(Sin GPU.)*

2. Lee la traza de la tarea del IVA (T04) con las reglas comprobadas al final. Reescribe el mensaje que la puerta le manda al agente con la R1 para que no le empuje a tocar el archivo de pruebas. ¿Basta con cambiar el mensaje, o hay que cambiar cuándo se comprueba la regla? *(GPU, unos 10 minutos.)*

3. Usa el `AGENTS.md` en prosa con las reglas comprobadas al actuar y en la puerta. Si las reglas se comprueban, ¿sigue importando cómo están escritas? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T04, T10 y T17. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

4. Escribe una regla R9, *no cambies la firma de una función que ya existe*, y su comprobación con el diff: busca líneas `def` quitadas y añadidas con el mismo nombre y distintos parámetros. ¿Es de proceso o de resultado? ¿Se podría comprobar al editar? *(Sin GPU.)*

5. Busca el `AGENTS.md` de tres proyectos de código abierto. Copia cinco instrucciones de cada uno y clasifícalas en comprobables y no comprobables. Para dos de las no comprobables, propón una versión que sí lo sea. *(Sin GPU.)*

6. Haz que la puerta final compruebe solo las reglas de resultado (R4, R7 y R8), sin comprobar nada al editar; en el programa es la opción `puerta="resultado"`. ¿Evita la trampa de la tarea del IVA? ¿Cuántas reglas de proceso se incumplen? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T04, T10 y T17. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

7. Aplica la expresión `PIDE_DOCS` a las veinte peticiones del banco y compara el resultado con las rutas permitidas de cada contrato, en `datos/tareas/Txx/alcance.json`. ¿Hay algún caso en que no coincidan? ¿Quién tiene razón? *(Sin GPU.)*

8. Con las reglas comprobadas al actuar, activa `exigir_marca=True`: una respuesta sin herramientas que no diga «TERMINADO» ni «NO TERMINADO» ya no acaba la tarea, y el agente sigue. ¿Cuántas tareas reclama ahora como terminadas? ¿Cuántas de ellas lo están? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T04, T10 y T17. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

9. Cuenta los tokens de los dos `AGENTS.md` con el tokenizador del modelo y multiplícalos por el número de llamadas al modelo de cada variante. ¿Qué parte de la diferencia de tokens entre la prosa y las reglas se debe al propio `AGENTS.md`? *(Sin GPU.)*

10. **Proyecto.** Si usas un agente de programación comercial que admita ganchos, como Claude Code, escribe un gancho que se ejecute antes de cada edición y la bloquee si el archivo está en una carpeta prohibida o no se ha leído antes en la sesión. Si no, añade a `HerramientasConReglas` una regla de proceso propia de tu repositorio y pruébala con tres tareas.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). La convención [AGENTS.md](https://agents.md). Los ganchos de Claude Code: [Hooks reference](https://code.claude.com/docs/en/hooks). La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.